# Smoke test the NIH single-CNN workflow

Check the attached NIH files, official list hashes, exact split hash, image availability, then run one real-image forward/backward/optimizer step for ConvNeXt-Tiny, DenseNet-121, and ResNet-50 using random initialization. This does not download pretrained weights or perform a full training run.

This notebook is self-contained: it uses the official NIH patient split, protocol validation checks, and hard labels only. No RAD-DINO teacher, distillation target, or distilled checkpoint is loaded.

## Kaggle setup

Attach NIH ChestX-ray14 as a notebook input and enable a GPU. Enable Internet for package installation and pretrained weights. Kaggle supplies CUDA-enabled PyTorch and torchvision; do not replace them.

In [ ]:
!pip install -q scikit-learn==1.6.1 timm==1.0.26


## Paths and options

Set `DATA_ROOT` to the NIH dataset directory containing `Data_Entry_2017.csv` and the official list files. Set `OUTPUT_DIR` under `/kaggle/working` so the outputs can be downloaded after the session.

In [ ]:
INPUT_DIR = "/kaggle/input"
DATA_ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"
OUTPUT_DIR = "/kaggle/working/single_model/output"
NUM_WORKERS = 4
BOOTSTRAP_RESAMPLES = 1000


## Shared data, model, and evaluation implementation

Run this cell after the setup and path cells. It verifies official list hashes, split counts, patient separation, and the protocol split hash before training.

In [ ]:
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import argparse
import csv
import gc
import hashlib
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset


LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia",
]
LABEL_ARRAY = np.asarray(LABELS)
EXPECTED_COUNTS = {"train": 73916, "val": 12608, "test": 25596}
EXPECTED_LIST_COUNTS = {"train_val_list.txt": 86524, "test_list.txt": 25596}
EXPECTED_LIST_HASHES = {
    "train_val_list.txt": "61fbe896321c1c1c8b75f3e4f3a08e4fef6486d95ef8a667c31d4d60dca6cb81",
    "test_list.txt": "38ca5ef7f756092946f57c1a59faca882ed589a1ab1f72590b45dc06c6d5e1cc",
}
EXPECTED_SPLIT_HASH = "b9f2fbc9aa061dafa76994494b92779651dc3ddd2ed3e7b82c70bbf5cf7071dd"
SEED = 42
IMG_SIZE = 384
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
MODEL_CONFIGS = {
    "convnext_tiny": {
        "model_id": "M08_CONVNEXT_TINY_ASL_384_CR",
        "timm_name": "convnext_tiny.fb_in22k_ft_in1k",
    },
    "densenet121": {
        "model_id": "M08_DENSENET121_ASL_384_CR",
        "timm_name": "densenet121.tv_in1k",
    },
    "resnet50": {
        "model_id": "M08_RESNET50_ASL_384_CR",
        "timm_name": "resnet50.tv_in1k",
    },
}


def _find_files(root, wanted):
    found = {name: [] for name in wanted}
    for current, _, files in os.walk(root):
        for name in wanted:
            if name in files:
                found[name].append(Path(current) / name)
        if all(found.values()):
            break
    return found


def _locate_data(input_dir, data_root=None):
    search_root = Path(input_dir).expanduser().resolve()
    if not search_root.exists():
        raise FileNotFoundError(f"Input directory does not exist: {search_root}")

    if data_root:
        root = Path(data_root).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"Data root does not exist: {root}")
        files = _find_files(root, {"Data_Entry_2017.csv", "train_val_list.txt", "test_list.txt"})
        missing = [name for name, paths in files.items() if not paths]
        if missing:
            raise FileNotFoundError(f"Missing under --data-root {root}: {', '.join(missing)}")
        metadata_paths = files["Data_Entry_2017.csv"]
    else:
        files = _find_files(
            search_root, {"Data_Entry_2017.csv", "train_val_list.txt", "test_list.txt"}
        )
        metadata_paths = files["Data_Entry_2017.csv"]
        if not metadata_paths:
            raise FileNotFoundError(
                f"Data_Entry_2017.csv not found below {search_root}; attach NIH ChestX-ray14 "
                "or pass --data-root."
            )
        if len(metadata_paths) != 1:
            raise ValueError(
                f"Found {len(metadata_paths)} metadata CSV files below {search_root}; "
                "pass --data-root to select the NIH dataset."
            )

    if len(metadata_paths) != 1:
        raise ValueError(
            f"Found {len(metadata_paths)} metadata CSV files below "
            f"{root if data_root else search_root}; select one dataset with --data-root."
        )
    metadata = metadata_paths[0]
    list_paths = {}
    for name in ("train_val_list.txt", "test_list.txt"):
        candidates = files[name]
        sibling = [path for path in candidates if path.parent == metadata.parent]
        if not candidates:
            raise FileNotFoundError(f"{name} not found below {root if data_root else search_root}")
        if sibling:
            list_paths[name] = sibling[0]
        elif len(candidates) == 1:
            list_paths[name] = candidates[0]
        else:
            raise ValueError(
                f"Found multiple {name} files; put the official lists beside the metadata "
                "or pass a narrower --data-root."
            )
    return metadata.parent, metadata, list_paths


def _read_ids(path):
    ids = Path(path).read_text(encoding="utf-8").split()
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate image IDs in {path}")
    return ids


def _sha256(data):
    return hashlib.sha256(data).hexdigest()


def load_protocol_data(input_dir, data_root=None):
    root, metadata_path, list_paths = _locate_data(input_dir, data_root)
    df = pd.read_csv(metadata_path)
    required = {"Image Index", "Patient ID", "Finding Labels"}
    missing_columns = required - set(df.columns)
    if missing_columns:
        raise ValueError(f"Metadata is missing columns: {sorted(missing_columns)}")
    if len(df) != 112120:
        raise ValueError(f"Expected 112120 metadata rows, found {len(df)}")

    df["Image Index"] = df["Image Index"].astype(str)
    df = df.sort_values("Image Index").reset_index(drop=True)
    findings = df["Finding Labels"].fillna("").str.split("|")
    for label in LABELS:
        df[label] = findings.map(lambda row, name=label: float(name in row))

    train_val_ids = _read_ids(list_paths["train_val_list.txt"])
    test_ids = _read_ids(list_paths["test_list.txt"])
    for name, ids in (
        ("train_val_list.txt", train_val_ids),
        ("test_list.txt", test_ids),
    ):
        if len(ids) != EXPECTED_LIST_COUNTS[name]:
            raise ValueError(f"QC-01: unexpected {name} row count: {len(ids)}")
        digest = _sha256("\n".join(sorted(ids)).encode("utf-8"))
        if digest != EXPECTED_LIST_HASHES[name]:
            raise ValueError(f"QC-01: {name} hash mismatch: {digest}")

    all_ids = set(df["Image Index"])
    if set(train_val_ids) & set(test_ids):
        raise ValueError("QC-01: official train/validation and test lists overlap")
    if set(train_val_ids) | set(test_ids) != all_ids:
        raise ValueError("QC-01: official lists do not exactly cover metadata images")

    split = np.where(df["Image Index"].isin(test_ids), "test", "trainval")
    tv_frame = df.loc[split == "trainval"]
    _, val_indices = next(
        GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(
            tv_frame, groups=tv_frame["Patient ID"]
        )
    )
    partition = np.asarray(split, dtype=object)
    partition[split == "trainval"] = "train"
    partition[tv_frame.index[val_indices]] = "val"
    df["partition"] = partition

    split_frame = (
        df[["Image Index", "Patient ID", "partition"]]
        .rename(
            columns={
                "Image Index": "image_index",
                "Patient ID": "patient_id",
            }
        )
        .sort_values("image_index")
    )
    split_text = split_frame.to_csv(index=False, lineterminator="\n")
    split_hash = _sha256(split_text.encode("utf-8"))
    if split_hash != EXPECTED_SPLIT_HASH:
        raise ValueError(
            "QC-03: official split hash mismatch. Check scikit-learn version; "
            f"expected {EXPECTED_SPLIT_HASH}, received {split_hash}."
        )

    counts = df["partition"].value_counts().to_dict()
    if counts != EXPECTED_COUNTS:
        raise ValueError(f"QC-01: split counts differ from protocol: {counts}")
    patients = {
        part: set(df.loc[df["partition"] == part, "Patient ID"])
        for part in ("train", "val", "test")
    }
    if (
        patients["train"] & patients["val"]
        or patients["train"] & patients["test"]
        or patients["val"] & patients["test"]
    ):
        raise ValueError("QC-02: a patient appears in multiple partitions")

    image_paths = {}
    for current, _, names in os.walk(root):
        for name in names:
            if name.endswith(".png"):
                image_paths.setdefault(name, str(Path(current) / name))
    df["path"] = df["Image Index"].map(image_paths)
    if df["path"].isna().any():
        missing_images = int(df["path"].isna().sum())
        raise FileNotFoundError(
            f"QC-04: {missing_images} metadata images have no PNG under {root}"
        )

    return df, split_hash, split_frame


class CXRDataset(Dataset):
    def __init__(self, frame, training=False, img_size=IMG_SIZE):
        self.frame = frame.reset_index(drop=True)
        self.labels = self.frame[LABELS].to_numpy(dtype=np.float32)
        self.transform = (
            T.Compose(
                [
                    T.RandomResizedCrop(
                        img_size,
                        scale=(0.85, 1.0),
                        ratio=(0.9, 1.1),
                        interpolation=T.InterpolationMode.BILINEAR,
                    ),
                    T.RandomHorizontalFlip(p=0.5),
                    T.RandomApply([T.RandomRotation(7)], p=0.5),
                    T.ColorJitter(brightness=0.1, contrast=0.1),
                    T.ToTensor(),
                ]
            )
            if training
            else T.Compose(
                [
                    T.Resize(
                        (img_size, img_size),
                        interpolation=T.InterpolationMode.BILINEAR,
                    ),
                    T.ToTensor(),
                ]
            )
        )
        self.normalize = T.Normalize(MEAN, STD)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["path"]) as image:
            image = image.convert("L")
            tensor = self.transform(image).repeat(3, 1, 1)
        return self.normalize(tensor), self.labels[index]


class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        probabilities = torch.sigmoid(logits.float())
        negative = (
            (probabilities - self.clip).clamp(min=0)
            if self.clip > 0
            else probabilities
        )
        positive_loss = (
            targets
            * torch.log(probabilities.clamp(min=self.eps))
            * (1 - probabilities).pow(self.gamma_pos)
        )
        negative_loss = (
            (1 - targets)
            * torch.log((1 - negative).clamp(min=self.eps))
            * negative.pow(self.gamma_neg)
        )
        return -(positive_loss + negative_loss).sum(dim=1).mean()


class CNNClassifier(nn.Module):
    def __init__(self, timm_name, pretrained=True):
        super().__init__()
        import timm

        if timm_name.startswith("densenet"):
            self.backbone = timm.create_model(
                timm_name, pretrained=pretrained, num_classes=0
            )
        else:
            self.backbone = timm.create_model(
                timm_name,
                pretrained=pretrained,
                num_classes=0,
                drop_path_rate=0.1,
            )
        self.head = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(self.backbone.num_features, len(LABELS)),
        )

    def forward(self, images):
        return self.head(self.backbone(images))


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def _worker_seed(_worker_id):
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed)
    random.seed(seed)


def _loader(frame, training, batch_size, workers):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        CXRDataset(frame, training=training),
        batch_size=batch_size,
        shuffle=training,
        num_workers=workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=workers > 0,
        drop_last=False,
        worker_init_fn=_worker_seed,
        generator=generator,
    )


def _macro_auroc(labels, logits):
    from sklearn.metrics import roc_auc_score

    values = []
    for index in range(len(LABELS)):
        target = labels[:, index]
        if target.min() != target.max():
            values.append(roc_auc_score(target, logits[:, index]))
    if not values:
        raise ValueError("Validation labels have no class with both positive and negative cases")
    return float(np.mean(values))


def _phase_optimizer(model, phase):
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.head.parameters():
        parameter.requires_grad = True
    groups = [
        {
            "params": list(model.head.parameters()),
            "lr": 1e-3 if phase == "warmup" else 1e-4,
        }
    ]
    if phase == "finetune":
        for parameter in model.backbone.parameters():
            parameter.requires_grad = True
        groups.append(
            {"params": list(model.backbone.parameters()), "lr": 1e-4}
        )
    return torch.optim.AdamW(groups, weight_decay=0.05)


def _schedule(optimizer, total_steps):
    warmup_steps = max(1, int(total_steps * 0.03))

    def scale(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        return 0.5 * (
            1 + math.cos(math.pi * (step - warmup_steps) / max(1, total_steps - warmup_steps))
        )

    return torch.optim.lr_scheduler.LambdaLR(optimizer, scale)


def _predict(model, loader, device):
    model.eval()
    logits = []
    labels = []
    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(device, non_blocking=True)
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=device.type == "cuda",
            ):
                output = model(images)
            logits.append(output.float().cpu().numpy())
            labels.append(targets.numpy())
    return np.concatenate(logits), np.concatenate(labels)


def _save_logits(path, frame, logits, labels, split_hash):
    np.savez_compressed(
        path,
        image_index=frame["Image Index"].to_numpy(dtype=str),
        patient_id=frame["Patient ID"].to_numpy(),
        logits=logits.astype(np.float32),
        labels=labels.astype(np.uint8),
        label_order=LABEL_ARRAY,
        split_sha256=np.asarray(split_hash),
    )


def _fit_platt(labels, logits):
    from scipy.optimize import minimize
    from scipy.special import expit

    parameters = np.zeros((len(LABELS), 2), dtype=np.float64)
    for index in range(len(LABELS)):
        target = labels[:, index].astype(np.float64)
        raw = logits[:, index].astype(np.float64)
        if target.min() == target.max():
            raise ValueError(f"Cannot calibrate {LABELS[index]}: validation labels are constant")

        def objective(coef):
            calibrated = coef[0] * raw + coef[1]
            probabilities = expit(calibrated)
            loss = np.logaddexp(0, calibrated) - target * calibrated
            residual = probabilities - target
            gradient = np.asarray([np.dot(residual, raw), residual.sum()])
            return float(loss.sum()), gradient

        result = minimize(
            objective,
            np.asarray([1.0, 0.0]),
            method="L-BFGS-B",
            jac=True,
        )
        if not result.success or not np.isfinite(result.fun):
            raise RuntimeError(f"Platt calibration failed for {LABELS[index]}: {result.message}")
        parameters[index] = result.x
    return parameters


def _calibrate(logits, parameters):
    from scipy.special import expit

    return expit(logits * parameters[:, 0] + parameters[:, 1])


def _fit_thresholds(labels, probabilities):
    thresholds = np.empty(len(LABELS), dtype=np.float64)
    fallback = []
    for index, name in enumerate(LABELS):
        target = labels[:, index].astype(np.int64)
        scores = probabilities[:, index]
        if target.sum() == 0:
            thresholds[index] = 0.5
            fallback.append(name)
            continue
        order = np.argsort(scores, kind="mergesort")[::-1]
        sorted_scores = scores[order]
        cumulative_tp = np.cumsum(target[order])
        ends = np.r_[np.flatnonzero(sorted_scores[:-1] != sorted_scores[1:]), len(scores) - 1]
        tp = cumulative_tp[ends]
        predicted = ends + 1
        fp = predicted - tp
        fn = target.sum() - tp
        denominator = 2 * tp + fp + fn
        f1 = np.divide(2 * tp, denominator, out=np.zeros_like(tp, dtype=float), where=denominator > 0)
        best = np.flatnonzero(f1 == f1.max())
        thresholds[index] = sorted_scores[ends[best[-1]]]
        if f1.max() == 0:
            thresholds[index] = 0.5
            fallback.append(name)
    return thresholds, fallback


def _metrics(labels, probabilities, thresholds, sample_weight=None):
    from sklearn.metrics import average_precision_score, roc_auc_score

    binary = probabilities >= thresholds[None, :]
    weights = (
        np.ones(len(labels), dtype=np.float64)
        if sample_weight is None
        else sample_weight.astype(np.float64)
    )
    weighted_labels = labels * weights[:, None]
    weighted_binary = binary * weights[:, None]
    tp = np.sum(weighted_labels * binary, axis=0)
    fp = np.sum((weights[:, None] - weighted_labels) * binary, axis=0)
    fn = np.sum(weighted_labels * (~binary), axis=0)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=tp + fp > 0)
    recall = np.divide(tp, tp + fn, out=np.full_like(tp, np.nan), where=tp + fn > 0)
    f1 = np.divide(
        2 * tp,
        2 * tp + fp + fn,
        out=np.zeros_like(tp),
        where=2 * tp + fp + fn > 0,
    )
    auroc = np.full(len(LABELS), np.nan, dtype=np.float64)
    auprc = np.full(len(LABELS), np.nan, dtype=np.float64)
    for index in range(len(LABELS)):
        target = labels[:, index]
        if np.unique(target[weights > 0]).size == 2:
            auroc[index] = roc_auc_score(
                target, probabilities[:, index], sample_weight=weights
            )
            auprc[index] = average_precision_score(
                target, probabilities[:, index], sample_weight=weights
            )
    total_tp, total_fp, total_fn = tp.sum(), fp.sum(), fn.sum()
    micro_precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else np.nan
    micro_f1 = (
        2 * total_tp / (2 * total_tp + total_fp + total_fn)
        if 2 * total_tp + total_fp + total_fn
        else 0.0
    )
    aggregates = {
        "macro_auroc": float(np.nanmean(auroc)),
        "macro_auprc": float(np.nanmean(auprc)),
        "macro_f1": float(np.mean(f1)),
        "micro_f1": float(micro_f1),
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.nanmean(recall)),
    }
    per_class = {
        name: {
            "auroc": float(auroc[index]),
            "auprc": float(auprc[index]),
            "precision": float(precision[index]),
            "recall": float(recall[index]),
            "f1": float(f1[index]),
            "support": int(labels[:, index].sum()),
        }
        for index, name in enumerate(LABELS)
    }
    return aggregates, per_class


def _patient_bootstrap(labels, probabilities, thresholds, patient_ids, seed=20260922, resamples=1000):
    unique_patients, patient_index = np.unique(patient_ids, return_inverse=True)
    rng = np.random.default_rng(seed)
    names = (
        "macro_auroc",
        "macro_auprc",
        "macro_f1",
        "micro_f1",
        "macro_precision",
        "macro_recall",
    )
    samples = {name: np.empty(resamples, dtype=np.float64) for name in names}
    for iteration in range(resamples):
        selected = rng.integers(0, len(unique_patients), size=len(unique_patients))
        patient_weights = np.bincount(selected, minlength=len(unique_patients))
        image_weights = patient_weights[patient_index]
        values, _ = _metrics(labels, probabilities, thresholds, image_weights)
        for name in names:
            samples[name][iteration] = values[name]
    return {
        name: {
            "low": float(np.nanpercentile(values, 2.5)),
            "high": float(np.nanpercentile(values, 97.5)),
        }
        for name, values in samples.items()
    }


def _dataset_characteristics(frame):
    positive_labels_per_image = frame[LABELS].sum(axis=1)
    details = {}
    for part in ("train", "val", "test"):
        subset = frame.loc[frame["partition"] == part]
        patients = subset["Patient ID"].nunique()
        views = subset["View Position"] if "View Position" in subset else None
        details[part] = {
            "images": int(len(subset)),
            "patients": int(patients),
            "mean_findings_per_image": float(
                positive_labels_per_image.loc[subset.index].mean()
            ),
            "pa_view_fraction": (
                float(views.eq("PA").mean()) if views is not None else None
            ),
            "prevalence": {
                name: float(subset[name].mean()) for name in LABELS
            },
        }
    return details


def train_model(model_key, args):
    cfg = MODEL_CONFIGS[model_key]
    seed_everything()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = False
    frame, split_hash, split_frame = load_protocol_data(args.input_dir, args.data_root)
    output_root = Path(args.output_dir).expanduser().resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    split_frame.to_csv(
        output_root / "split_NIH_OFFICIAL_PV.csv",
        index=False,
        lineterminator="\n",
    )

    run_name = f"NIH_CXR14__NIH_OFFICIAL_PV__{cfg['model_id']}__s{SEED}"
    run_dir = output_root / "runs" / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    config = {
        "model_id": cfg["model_id"],
        "timm_name": cfg["timm_name"],
        "dataset_id": "NIH_CXR14",
        "split_id": "NIH_OFFICIAL_PV",
        "protocol_version": "2.2.0",
        "split_sha256": split_hash,
        "seed": SEED,
        "label_order": LABELS,
        "img_size": IMG_SIZE,
        "batch_size": 32,
        "warmup_epochs": 1,
        "finetune_epochs": 8,
        "patience": 2,
        "lr_head_warmup": 1e-3,
        "lr_head": 1e-4,
        "lr_backbone": 1e-4,
        "weight_decay": 0.05,
        "drop_path": None if model_key == "densenet121" else 0.1,
        "dropout": 0.2,
        "asl": {"gamma_neg": 4, "gamma_pos": 1, "clip": 0.05},
        "augmentation": {
            "random_resized_crop_scale": [0.85, 1.0],
            "random_resized_crop_ratio": [0.9, 1.1],
            "horizontal_flip_p": 0.5,
            "rotation_degrees": 7,
            "rotation_p": 0.5,
            "color_jitter_brightness": 0.1,
            "color_jitter_contrast": 0.1,
        },
        "normalization": {"mean": MEAN, "std": STD},
        "amp": "fp16 on CUDA",
        "tta": "none",
        "device": str(device),
        "pretrained": True,
    }
    (run_dir / "config.json").write_text(
        json.dumps(config, indent=2), encoding="utf-8"
    )
    train_frame = frame.loc[frame["partition"] == "train"].reset_index(drop=True)
    val_frame = frame.loc[frame["partition"] == "val"].reset_index(drop=True)
    test_frame = frame.loc[frame["partition"] == "test"].reset_index(drop=True)
    train_loader = _loader(train_frame, True, 32, args.workers)
    val_loader = _loader(val_frame, False, 32, args.workers)

    model = CNNClassifier(cfg["timm_name"], pretrained=True).to(device)
    loss_fn = AsymmetricLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    best_score = -np.inf
    best_epoch = 0
    best_path = run_dir / "checkpoint_best.pt"
    training_log = []
    bad_finetune_epochs = 0
    epoch_number = 0
    stopped_early = False

    for phase, epochs in (("warmup", 1), ("finetune", 8)):
        optimizer = _phase_optimizer(model, phase)
        scheduler = _schedule(optimizer, len(train_loader) * epochs)
        if phase == "finetune":
            bad_finetune_epochs = 0
        for phase_epoch in range(epochs):
            epoch_number += 1
            epoch_start = time.time()
            model.train()
            running_loss = 0.0
            samples = 0
            for images, targets in train_loader:
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.float16,
                    enabled=device.type == "cuda",
                ):
                    logits = model(images)
                    loss = loss_fn(logits, targets)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite training loss at epoch {epoch_number}")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                running_loss += float(loss.detach()) * len(images)
                samples += len(images)

            val_logits, val_labels = _predict(model, val_loader, device)
            score = _macro_auroc(val_labels, val_logits)
            improved = score > best_score
            if improved:
                best_score = score
                best_epoch = epoch_number
                checkpoint = {
                    "model_state": model.state_dict(),
                    "model_id": cfg["model_id"],
                    "split_sha256": split_hash,
                    "best_epoch": best_epoch,
                    "best_val_macro_auroc": best_score,
                }
                temporary = best_path.with_suffix(".pt.tmp")
                torch.save(checkpoint, temporary)
                os.replace(temporary, best_path)
                bad_finetune_epochs = 0
            elif phase == "finetune":
                bad_finetune_epochs += 1

            training_log.append(
                {
                    "epoch": epoch_number,
                    "phase": phase,
                    "phase_epoch": phase_epoch + 1,
                    "train_loss": running_loss / max(1, samples),
                    "val_macro_auroc": score,
                    "best_val_macro_auroc": best_score,
                    "epoch_seconds": time.time() - epoch_start,
                }
            )
            print(
                f"{cfg['model_id']} epoch {epoch_number}: "
                f"loss={running_loss / max(1, samples):.5f} "
                f"val_macro_auroc={score:.5f} best={best_score:.5f}",
                flush=True,
            )
            if phase == "finetune" and bad_finetune_epochs >= 2:
                stopped_early = True
                break
        if stopped_early:
            break

    with (run_dir / "train_log.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=training_log[0].keys())
        writer.writeheader()
        writer.writerows(training_log)
    checkpoint = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])

    # Freeze all validation-based decisions before making test predictions.
    val_logits, val_labels = _predict(model, val_loader, device)
    calibration = _fit_platt(val_labels, val_logits)
    val_probabilities = _calibrate(val_logits, calibration)
    thresholds, fallback_labels = _fit_thresholds(val_labels, val_probabilities)
    (run_dir / "calibration.json").write_text(
        json.dumps(
            {
                "method": "per-class Platt scaling; validation only",
                "label_order": LABELS,
                "parameters": [
                    {"a": float(a), "b": float(b)} for a, b in calibration
                ],
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    (run_dir / "thresholds.json").write_text(
        json.dumps(
            {
                "method": "per-class validation F1 maximization; ties choose smallest threshold",
                "label_order": LABELS,
                "thresholds": {
                    name: float(thresholds[index])
                    for index, name in enumerate(LABELS)
                },
                "f1_zero_fallback_labels": fallback_labels,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    test_loader = _loader(test_frame, False, 32, args.workers)
    test_logits, test_labels = _predict(model, test_loader, device)
    _save_logits(run_dir / "logits_val.npz", val_frame, val_logits, val_labels, split_hash)
    _save_logits(
        run_dir / "logits_test.npz", test_frame, test_logits, test_labels, split_hash
    )
    test_probabilities = _calibrate(test_logits, calibration)
    metrics, per_class = _metrics(test_labels, test_probabilities, thresholds)
    intervals = _patient_bootstrap(
        test_labels,
        test_probabilities,
        thresholds,
        test_frame["Patient ID"].to_numpy(),
        resamples=args.bootstrap_resamples,
    )
    results = {
        "model_id": cfg["model_id"],
        "split_id": "NIH_OFFICIAL_PV",
        "split_sha256": split_hash,
        "seed": SEED,
        "tta": "none",
        "calibration": "per-class Platt scaling fitted on validation",
        "thresholds": "per-class F1-maximizing validation thresholds",
        "best_epoch": best_epoch,
        "best_val_macro_auroc": best_score,
        "epochs_trained": len(training_log),
        "stopped_early": stopped_early,
        "test_metrics": metrics,
        "test_metrics_95pct_patient_bootstrap_ci": intervals,
        "per_class_test_metrics": per_class,
        "bootstrap": {
            "method": "patient-level percentile bootstrap",
            "resamples": args.bootstrap_resamples,
            "seed": 20260922,
        },
        "dataset_characteristics": _dataset_characteristics(frame),
    }
    (run_dir / "test_metrics.json").write_text(
        json.dumps(results, indent=2, allow_nan=True), encoding="utf-8"
    )
    print(f"Run complete: {run_dir}", flush=True)


def smoke_test(args):
    frame, split_hash, _ = load_protocol_data(args.input_dir, args.data_root)
    train_frame = frame.loc[frame["partition"] == "train"].head(2).reset_index(drop=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(
        f"Protocol data checks passed: split={split_hash}; "
        f"device={device}; real NIH images={len(train_frame)}",
        flush=True,
    )

    for model_key, cfg in MODEL_CONFIGS.items():
        seed_everything()
        dataset = CXRDataset(train_frame, training=True)
        images = torch.stack([dataset[index][0] for index in range(len(dataset))]).to(device)
        targets = torch.from_numpy(dataset.labels).to(device)
        model = CNNClassifier(cfg["timm_name"], pretrained=False).to(device)
        model.train()
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model(images)
            loss = AsymmetricLoss()(logits, targets)
        if logits.shape != (len(train_frame), len(LABELS)):
            raise AssertionError(f"{model_key}: incorrect logits shape {tuple(logits.shape)}")
        if not torch.isfinite(logits).all() or not torch.isfinite(loss):
            raise FloatingPointError(f"{model_key}: non-finite forward pass or loss")
        loss.backward()
        if not any(
            parameter.grad is not None and torch.isfinite(parameter.grad).all()
            for parameter in model.parameters()
        ):
            raise FloatingPointError(f"{model_key}: no finite gradients")
        optimizer.step()
        print(
            f"{model_key}: real-image transform, forward, ASL, backward, and optimizer step passed",
            flush=True,
        )
        del model, optimizer, images, targets, dataset
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    print("All single-CNN smoke tests passed.", flush=True)


def parse_args(model_key=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-dir", default="/kaggle/input")
    parser.add_argument("--data-root", default=None)
    parser.add_argument(
        "--output-dir", default="/kaggle/working/single_model/output"
    )
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--bootstrap-resamples", type=int, default=1000)
    if model_key is None:
        parser.add_argument(
            "--models",
            nargs="+",
            choices=(*MODEL_CONFIGS.keys(), "all"),
            default=["all"],
        )
    return parser.parse_args()


def train_entry(model_key):
    args = parse_args(model_key)
    if args.workers < 0:
        raise ValueError("--workers must be zero or greater")
    if args.bootstrap_resamples < 1:
        raise ValueError("--bootstrap-resamples must be greater than zero")
    train_model(model_key, args)


def smoke_entry():
    args = parse_args()
    smoke_test(args)



## Run

Check the attached NIH files, official list hashes, exact split hash, image availability, then run one real-image forward/backward/optimizer step for ConvNeXt-Tiny, DenseNet-121, and ResNet-50 using random initialization. This does not download pretrained weights or perform a full training run.

In [ ]:
notebook_args = argparse.Namespace(input_dir=INPUT_DIR, data_root=DATA_ROOT, output_dir=OUTPUT_DIR, workers=NUM_WORKERS, bootstrap_resamples=BOOTSTRAP_RESAMPLES)
smoke_test(notebook_args)
